# 01. 학습·테스트 데이터셋 수집

FFHQ 실제 얼굴과 StyleGAN2/3 생성 얼굴을 모아 학습 데이터셋의 뼈대를 만든 노트북.

**핵심 설계 — 학습/테스트 데이터 분리**

마지막 셀에서 FFHQ 스트리밍에 `dataset.skip(40000).take(12000)`을 적용했다.
학습에 쓴 앞 40,000장을 건너뛰고 그다음 12,000장만 테스트용 Real로 가져와,
**학습 데이터가 테스트에 섞이는 것(data leakage)을 구조적으로 차단**했다.

- 실제(Real): HuggingFace `marcosv/ffhq-dataset` 스트리밍, 이어받기 지원
- 가짜(Fake): StyleGAN3 20,000장 / StyleGAN2 20,000장


In [ ]:
import os
from google.colab import drive

# 1. 아주 깨끗한 상태로 드라이브 연결
print("🔌 구글 드라이브 연결 중...")
drive.mount('/content/drive')

# 2. 프로젝트 폴더 뼈대 생성
#base_dir = '/content/drive/MyDrive/WayV_Detection_Train'
#sub_dirs = ['Real/FFHQ', 'Fake/StyleGAN2', 'Fake/StyleGAN3']

#for d in sub_dirs:
   # os.makedirs(f'{base_dir}/{d}', exist_ok=True)

print("✅ 'WayV_Detection_Train' 구조 세팅 완료!")

🔌 구글 드라이브 연결 중...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 'WayV_Detection_Train' 구조 세팅 완료!


In [ ]:
# 1. 대상 폴더 생성 (이미 있으면 통과)
!mkdir -p /content/drive/MyDrive/WayV_Detection_Train/Fake/StyleGAN3

# 2. 허깅페이스에서 images.zip 파일만 다이렉트로 고속 다운로드 (약 2.1GB)
print("🌍 허깅페이스에서 StyleGAN3 사진 압축파일만 빼오는 중...")
!wget -q --show-progress "https://huggingface.co/datasets/balgot/stylegan3-annotated/resolve/main/images.zip" -O /content/sg3_images.zip

# 3. 다운받은 압축 파일을 WayV 프로젝트 폴더에 시원하게 해제!
print("💥 다운로드 완료! WayV 폴더에 사진 압축 푸는 중...")
!unzip -q -j /content/sg3_images.zip -d /content/drive/MyDrive/WayV_Detection_Train/Fake/StyleGAN3/

# 4. 다 쓴 임시 압축 파일 삭제
!rm /content/sg3_images.zip

print("\n🎉 StyleGAN3 데이터 2만 장 수집이 깔끔하게 끝났습니다!")

🌍 허깅페이스에서 StyleGAN3 사진 압축파일만 빼오는 중...
💥 다운로드 완료! WayV 폴더에 사진 압축 푸는 중...

🎉 StyleGAN3 데이터 2만 장 수집이 깔끔하게 끝났습니다!


In [ ]:
# 1. 내 드라이브에 만든 '바로가기'를 꽉 묶어서 압축합니다. (약 2분 소요)
print("📦 공유받은 사진 2만 장을 코랩으로 빨아들이는 중...")
!zip -r -q /content/temp_gan2.zip /content/drive/MyDrive/StyleGAN2_Dataset_20k/*

# 2. 압축된 진짜 파일들을 WayV 프로젝트 폴더에 시원하게 풉니다.
print("💥 WayV 프로젝트 폴더에 진짜 파일로 압축 해제 중...")
!unzip -q -j /content/temp_gan2.zip -d /content/drive/MyDrive/WayV_Detection_Train/Fake/StyleGAN2/

# 3. 다 쓴 임시 압축 파일은 삭제하여 용량을 아낍니다.
!rm /content/temp_gan2.zip

print("\n🎉 StyleGAN2 데이터 2만 장 이동이 완벽하게 완료되었습니다!")

📦 공유받은 사진 2만 장을 코랩으로 빨아들이는 중...


zip error: Interrupted (aborting)
💥 WayV 프로젝트 폴더에 진짜 파일로 압축 해제 중...
unzip:  cannot find or open /content/temp_gan2.zip, /content/temp_gan2.zip.zip or /content/temp_gan2.zip.ZIP.
rm: cannot remove '/content/temp_gan2.zip': No such file or directory

🎉 StyleGAN2 데이터 2만 장 이동이 완벽하게 완료되었습니다!


In [ ]:
import os
import shutil
from tqdm.auto import tqdm

# 1. 경로 설정
source_dir = '/content/drive/MyDrive/StyleGAN2_Dataset_20k'
target_dir = '/content/drive/MyDrive/WayV_Detection_Train/Fake/StyleGAN2'

# 2. 딱 가져올 폴더만 타겟팅!
target_batches = ['batch_7', 'batch_8']
files_to_move = []

print("🔍 남은 배치(7, 8) 안의 파일만 쏙쏙 찾아내는 중...")
for batch_name in target_batches:
    batch_path = os.path.join(source_dir, batch_name)
    if os.path.exists(batch_path):
        for file in os.listdir(batch_path):
            if file.endswith('.png'):
                files_to_move.append(os.path.join(batch_path, file))
    else:
        print(f"⚠️ {batch_name} 폴더를 찾을 수 없습니다. 원본 폴더 안에 이 이름이 있는지 확인해주세요.")

total_files = len(files_to_move)
print(f"🎯 총 {total_files}장의 남은 사진을 찾았습니다! 이동 시작합니다.\n")

# 3. 안전하게 복사 진행
success_count = 0
skip_count = 0

for file_path in tqdm(files_to_move, desc="🚀 남은 배치 복사 진행률"):
    file_name = os.path.basename(file_path)
    target_path = os.path.join(target_dir, file_name)

    # 이미 목표 폴더에 들어있는 사진이면 0.1초 만에 건너뜀
    if os.path.exists(target_path):
        skip_count += 1
        continue

    try:
        shutil.copy(file_path, target_path)
        success_count += 1
    except Exception as e:
        pass

print("\n" + "="*40)
print(f"✅ 새로 옮긴 사진: {success_count}장")
print(f"⏭️ 이미 있어서 건너뛴 사진: {skip_count}장")
print("🎉 드디어 StyleGAN2 2만 장 완전체가 되었습니다!")

🔍 남은 배치(7, 8) 안의 파일만 쏙쏙 찾아내는 중...
🎯 총 5000장의 남은 사진을 찾았습니다! 이동 시작합니다.




✅ 새로 옮긴 사진: 4974장
⏭️ 이미 있어서 건너뛴 사진: 26장
🎉 드디어 StyleGAN2 2만 장 완전체가 되었습니다!


In [ ]:
import os
from datasets import load_dataset
from tqdm.auto import tqdm

# 1. 경로 설정
base_dir = '/content/drive/MyDrive/WayV_Detection_Train'
ffhq_dir = f'{base_dir}/Real'
os.makedirs(ffhq_dir, exist_ok=True)

print("🚨 엔비디아 공식 서버 다운 감지! 허깅페이스 우회 경로로 접속합니다...")
print("🌍 FFHQ (Real) 사진 4만 장 다이렉트 스트리밍 시작...\n")

# 2. 허깅페이스 데이터셋 로드 (메모리 폭발 방지를 위해 streaming=True)
try:
    # 가장 안정적인 FFHQ 미러 데이터셋을 우선 사용합니다.
    dataset = load_dataset("marcosv/ffhq-dataset", split="train", streaming=True)
except Exception:
    # 혹시 모를 상황을 대비한 예비 데이터셋
    dataset = load_dataset("student/FFHQ", split="train", streaming=True)

success_count = 0
target_count = 40000

# 3. 4만 장 다운로드 시작
with tqdm(total=target_count, desc="FFHQ 4만 장 저장 중") as pbar:
    for i, item in enumerate(dataset):
        if success_count >= target_count:
            break

        save_path = f"{ffhq_dir}/ffhq_real_{i:05d}.png"

        # 🚀 핵심: 이미 드라이브에 있는 사진은 0.1초 만에 건너뜀 (이어받기)
        if os.path.exists(save_path):
            success_count += 1
            pbar.update(1)
            continue

        try:
            # 허깅페이스 데이터의 'image' 객체를 로컬 파일로 바로 저장
            item['image'].save(save_path)
            success_count += 1
            pbar.update(1)
        except Exception:
            # 손상된 이미지가 섞여 있을 경우 무시하고 다음으로 넘어감
            pass

print("\n🎉 드디어 길고 길었던 데이터 수집이 모두 끝났습니다! 8만 장 세팅 완벽합니다!")

🚨 엔비디아 공식 서버 다운 감지! 허깅페이스 우회 경로로 접속합니다...
🌍 FFHQ (Real) 사진 4만 장 다이렉트 스트리밍 시작...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



🎉 드디어 길고 길었던 데이터 수집이 모두 끝났습니다! 8만 장 세팅 완벽합니다!


In [ ]:
import os
from datasets import load_dataset
from tqdm.auto import tqdm

# 1. 경로 설정
base_test_dir = '/content/drive/MyDrive/WayV_Detection_Test'
targets = [
    {'name': 'FLUX1_dev', 'path': f'{base_test_dir}/FLUX1_dev/Real'},
    {'name': 'FLUX1_schnell', 'path': f'{base_test_dir}/FLUX1_schnell/Real'},
    {'name': 'SDXL', 'path': f'{base_test_dir}/SDXL/Real'}
]

for t in targets:
    os.makedirs(t['path'], exist_ok=True)

print("🌍 허깅페이스 서버에 접속 중... (경고문구는 무시하세요)")

try:
    dataset = load_dataset("marcosv/ffhq-dataset", split="train", streaming=True)
except Exception:
    dataset = load_dataset("student/FFHQ", split="train", streaming=True)

# 🚀 핵심: 파이썬 for문이 아니라 허깅페이스 자체 기능으로 4만 장을 단숨에 스킵합니다!
print("⏳ 앞서 사용한 4만 장을 고속으로 스킵하는 중... (서버에서 찾는 데 약 1~2분 소요)")
dataset = dataset.skip(40000).take(12000)

# 2. 중복 제로 분배 시작
# 이제 dataset에는 정확히 4만 번 ~ 5.2만 번 사진만 1.2만장 들어있습니다.
with tqdm(total=12000, desc="Test용 Real 사진 분배 중") as pbar:
    for i, item in enumerate(dataset):

        real_index = i + 40000 # 실제 파일명에 각인될 원본 번호 (40000번부터 시작)

        # 3. 목적지 폴더 3등분
        if i < 4000:
            target = targets[0] # FLUX1_dev (0~3999)
        elif i < 8000:
            target = targets[1] # FLUX1_schnell (4000~7999)
        else:
            target = targets[2] # SDXL (8000~11999)

        save_path = f"{target['path']}/ffhq_real_{target['name']}_{real_index:05d}.png"

        # 이어받기 기능
        if os.path.exists(save_path):
            pbar.update(1)
            continue

        try:
            item['image'].save(save_path)
            pbar.update(1)
        except Exception:
            pass

print("\n🎉 고속 스킵 완료! 3개의 Test 폴더에 무결점으로 분배되었습니다!")

🌍 허깅페이스 서버에 접속 중... (경고문구는 무시하세요)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


⏳ 앞서 사용한 4만 장을 고속으로 스킵하는 중... (서버에서 찾는 데 약 1~2분 소요)



🎉 고속 스킵 완료! 3개의 Test 폴더에 무결점으로 분배되었습니다!
